# AA test tutorial 
AA test is important part of randomized controlled experiment, for example AB test. 

The objectives of the AA test are to verify the assumption of uniformity of samples as a result of the applied partitioning method, to select the best partition from the available ones, and to verify the applicability of statistical criteria for checking uniformity. 

For example, there is a hypothesis about the absence of dependence of features on each other. If this hypothesis is not followed, the AA test will fail.

[Wiki AA test](https://github.com/sb-ai-lab/HypEx/wiki/%D0%90%D0%90-Test) with more detailed description of terms for AA test.

<ul>
  <li><a href="#creation-of-a-new-test-dataset-with-synthetic-data">Creation of a new test dataset with synthetic data.
  <li><a href="#one-split-of-aa-test">One split of AA test.
  <li><a href="#aa-test">AA test.
  <li><a href="#aa-test-with-stratification">AA test with stratification.
</ul>

In [1]:
from hypex import AATest
from hypex.dataset import (
    ConstGroupRole,
    Dataset,
    InfoRole,
    StratificationRole,
    TargetRole,
    TreatmentRole,
)
from hypex.utils import create_test_data

ModuleNotFoundError: No module named 'pyspark'

## Creation of a new test dataset with synthetic data. 

In order to be able to work with our data in HypEx, first we need to convert it into `dataset`. It is important to mark the data fields by assigning the appropriate `roles`:
- TargetRole: a role for columns that contain features or predictor variables. Our split will be based on them. Applied by default if the role is not specified for the column.
- TreatmentRole: a role for columns that show the treatment or intervention.
- InfoRole: a role for columns that contain information about the data, such as user IDs. 

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
    }, data=create_test_data(),
)
data

In [ ]:
data.roles

## AA test
Then we run the experiment on our prepared dataset, wrapped into ExperimentData. In this case we select one of the pre-assembled pipeline, AA_TEST.
We can set the number of iterations for simple execution. In this case the random states are the numbers of each iteration.

In [ ]:
test = AATest(n_iterations=10)
result = test.execute(data)

In [ ]:
result.resume

**Interpretation of AA test results**

Each row in the table corresponds to a target feature being tested for equality between the control and test groups. Two statistical tests are used:

- **TTest**: tests if means are statistically different.
- **KSTest**: tests if distributions differ.

The `OK` / `NOT OK` labels show whether the difference is statistically significant. A `NOT OK` result indicates a possible imbalance.

Typical threshold:
- If p-value < 0.05 → `NOT OK` (statistically significant difference)
- If p-value ≥ 0.05 → `OK` (no significant difference)

If any metric has a `NOT OK` status in the `AA test` column, it means at least one iteration showed significant difference.


In [ ]:
result.aa_score

**Interpreting `aa_score`**

This output shows p-values and the overall pass/fail status for each test type and feature. A high p-value (close to 1.0) means the test passed — the groups are similar.

- `score`: p-value of the statistical test.
- `pass`: True if no iterations showed significant differences.

Note: Even if the average p-value is high, the `pass` might still be False if at least one of the iterations had a p-value < 0.05.


In [ ]:
result.best_split

**About `best_split`**

This shows the best found split of the dataset, where control and test groups are as similar as possible in terms of target metrics.

You can use this split for future modeling or as a validation check before proceeding to actual experiments.


In [ ]:
result.best_split_statistic

**Understanding `best_split_statistic`**

This table contains detailed statistics for the best (most balanced) split found across all iterations. You can compare:

- Mean values in control vs test group.
- Absolute and relative differences.
- p-values for both tests.

Ideally, all rows should have `OK` in both TTest and KSTest columns, and small difference values (<1%).

In [ ]:
result.experiments

# AA Test with random states

We can also adjust some of the preset parameters of the experiment by assigning them to the respective params of the experiment. I.e. here we set the range of the random states we want to run our AA test for. 

In [ ]:
test = AATest(random_states=[56, 72, 2, 43])
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AA Test with stratification

Depending on your requirements it is possible to stratify the data. You can set `stratification=True` and `StratificationRole` in `Dataset` to run it with stratification.

Stratified AA tests ensure that both groups (control/test) have the same proportions of categories (e.g. same % of genders or regions). This prevents imbalances in categorical features that can distort results.

Make sure to assign `StratificationRole` to relevant columns in your dataset before enabling stratification.

In [ ]:
test = AATest(random_states=[56, 72, 2, 43], stratification=True)
result = test.execute(data)

100%|██████████| 4/4 [00:01<00:00,  2.25it/s]


In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AA Test by samples 

Depending on your requirements and size of data it is possible to estimate AA test on samples the data. You can set `sample_size=size` to run it. 

In [ ]:
test = AATest(n_iterations=10, sample_size=0.3)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AATest with Target Role for a categorical feature

It is possible to assign Target Role to categorical features. A categorical feature can also be the target or outcome variable. In this case, the Chi-square test is added to the pipeline of AATest.

In [ ]:
data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "treat": TreatmentRole(int),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": TargetRole(str)
    }, data=create_test_data(),
)
data

In [ ]:
test = AATest(n_iterations=10)
result = test.execute(data)

In [ ]:
result.resume

In [ ]:
result.aa_score

In [ ]:
result.best_split

In [ ]:
result.best_split_statistic

In [ ]:
result.experiments

# AATest with unequal group sizes

AATest can be performed to get a split with unequal the groups of different sizes by using `unequal_size` argument. Also Whelch correction can be applied by adding `t_test_equal_vat=False` argument while initiating AATest instance.

In [ ]:
test = AATest(n_iterations=10, control_size=0.3, t_test_equal_var=False)
result = test.execute(data)

100%|██████████| 10/10 [00:04<00:00,  2.25it/s]


In [ ]:
result.best_split.data.groupby("split").agg("count")

In [ ]:
result.best_split_statistic

# AAnTest

AAnTest is an extension of AATest that allows to split the dataset into several test groups, additionally to the control group.

In [ ]:
test = AATest(groups_sizes=[0.3, 0.2, 0.2, 0.3])
result = test.execute(data)

In [ ]:
result.best_split.data.groupby("split").agg("count")

In [ ]:
result.best_split_statistic

# AATest with partially pre-defined groups

Certain users can be pre-assigned to either the test or the control group, so that they are not randomly assigned. This can be done using the `ConstGroupRole` role. In order to pre-assign users to the control group they should have a value of `control`, and in the test group they should have a value of `test` in the column with the role `ConstGroupRole`. Users that are not pre-assigned to either the control or the test group shoul have `None`, so that they will be assigned randomly.

In [ ]:
pd_data= create_test_data()
pd_data.loc[pd_data["treat"]==0, "const_grp"] = "control"
pd_data.loc[pd_data["treat"]==1, "const_grp"] = "test"
pd_data.loc[2000:, "const_grp"] = None

data = Dataset(
    roles={
        "user_id": InfoRole(int),
        "const_grp": ConstGroupRole(str),
        "pre_spends": TargetRole(),
        "post_spends": TargetRole(),
        "gender": StratificationRole(str),
        "industry": TargetRole(str),
    }, data=pd_data,
)
data

In [ ]:
test = AATest(n_iterations=1)
result = test.execute(data)

100%|██████████| 1/1 [00:00<00:00,  1.41it/s]


In [ ]:
result.resume

In [ ]:
result.best_split

## Common issues and tips

- **Missing roles**: Make sure all target variables are assigned `TargetRole`. Columns without roles may cause silent failure.
- **Stratification**: If your dataset contains categorical features (e.g. `gender`, `region`) that may affect the outcome, use `StratificationRole` and enable `stratification=True` in `AATest(...)`.
- **Imbalanced categories**: If some categories have too few samples, stratified splits may become unstable. Consider filtering or merging rare categories.
- **Random fluctuations**: On small datasets, it's normal to see occasional `NOT OK` results. Use more iterations (e.g. `n_iterations=50`) for stability.
- **Missing values**: NaNs in stratification columns may be treated as separate categories. Clean or fill missing values before stratified AA tests.